# Optuna — walk-forward housing model tuning

A dedicated **development-only** hyperparameter notebook for your SSU Data Dash repository.

- Tune one **model family × track** per study: `catboost`, `lightgbm`, `xgboost` × `off_market`, `list_aware`.
- Score pooled **MdAPE** from chronological **September–November 2025** folds (editable); save per-month MdAPE, MAPE, and bias.
- **Never use December 2025–May 2026 for Optuna trials.** Reserve it for limited, separately run confirmation in `model_run.ipynb`.
- Cache engineered features **once per backend, track, and month**; each trial trains only the estimator. This matters because historical comparable features are expensive.
- Persist studies in SQLite, support pause/resume, enqueue the existing configuration as a baseline, and prune after intermediate folds.
- Keep the repository's **single-stage additive-log model semantics**: off-market predicts (optionally centered) `log(sold_price)`; list-aware predicts `log(sold_price / list_price)`.

> **Factory integration:** This notebook produces `MODEL_SPECS` and includes an optional factory parity check. It does **not** change your existing `models/model.py` or `models/direct_listing.py`. When you refactor the factory, update only `build_factory_model` below (or import your new factory there), then run the parity check before trusting an Optuna study.

## 0 · Environment

Run from the repository root using your project's `.venv` notebook kernel. With `uv`:

```powershell
uv add optuna
uv sync
uv run jupyter lab
```

If the notebook kernel isn't selected, pick the Python interpreter under `.venv`. You can install the kernel with `uv run python -m ipykernel install --sys-prefix --name ssu-data-dash --display-name "Python (SSU Data Dash)"`.

This notebook assumes `data.py`, `models/`, and `model_run.ipynb` are in the current project root. No notebook imports or executes `model_run.ipynb`.

In [1]:
from __future__ import annotations

# Locate the checkout from Jupyter/VS Code, including nested notebook folders.
from pathlib import Path
import os
import sys
_start = Path(os.environ.get("DATA_DASH_ROOT", Path.cwd())).expanduser().resolve()
ROOT = next((p for p in (_start, *_start.parents)
             if (p / "data.py").is_file() and (p / "models").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Set DATA_DASH_ROOT to your Data Dash checkout before running this notebook.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


from copy import deepcopy
from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd
import optuna

import data
from features.features import make_features
from models.boost_regressor import CatBoostRegressor, LightGBMRegressor, XGBoostRegressor
from models.model import OffMarketModel
from models.direct_listing import DirectListingModel

ROOT = Path.cwd().resolve()
assert (ROOT / "models" / "features.py").exists(), (
    f"Open optuna.ipynb from the repository root, not {ROOT}"
)
print("Repository:", ROOT)
print("Optuna:", optuna.__version__)

Repository: C:\Users\deant\Desktop\Coding\Data Dash Case Comp
Optuna: 5.0.0


## 1 · Experiment configuration

Choose **one** backend and track per run. Start with `lightgbm` + `off_market`. Different backends have different categorical preprocessing, so **change the backend and rerun the entire notebook** to rebuild the cached matrices.

`OFF_MARKET_CENTER_TARGET=True` matches the `OffMarketModel(center_target=True)` call currently shown in your main notebook. If you change that factory setting, change it here too. For list-aware, keep the ratio target matching your existing `DirectListingModel(target="ratio")`.

The default tuning months **precede** December 2025. All features and targets are fitted on the available earlier sales for each month. `N_TRIALS` is the number of **additional** trials on this run, not the total. Before a large study, try `N_TRIALS=2`.

**Validation mode:** `expanding` retrains before each development month. `fixed_origin` trains once before the first development month, then evaluates all months without updating; this resembles the hidden June–September horizon more closely. Run **separate Optuna studies** for each mode.


In [11]:
BACKEND = "catboost"      # "lightgbm", "xgboost", "catboost"
TRACK = "off_market"      # "off_market" or "list_aware"
FEATURE_NAME = "Xpt_spatiotemporal"
DEV_MONTHS = ("2025-09", "2025-10", "2025-11")
CV_MODE = "expanding"     # or "fixed_origin": one model frozen before first DEV_MONTHS month
N_TRIALS = 20
STUDY_VERSION = "v1"      # bump when changing objective/feature definition/search space
SEED = 42
N_JOBS_STUDIES = 1        # avoid nested all-core parallelism

OFF_MARKET_CENTER_TARGET = True
LIST_AWARE_TARGET = "ratio"       # same target as current DirectListingModel

RESULTS_DIR = ROOT / "tuning"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
STORAGE = f"sqlite:///{(RESULTS_DIR / 'optuna_studies.db').as_posix()}"

assert CV_MODE in {"expanding", "fixed_origin"}
assert BACKEND in {"lightgbm", "xgboost", "catboost"}
assert TRACK in {"off_market", "list_aware"}
assert LIST_AWARE_TARGET == "ratio", "This notebook currently tunes the list-aware ratio target only"
periods = tuple(pd.Period(m, freq="M") for m in DEV_MONTHS)
assert len(periods) >= 2 and len(set(periods)) == len(periods)
assert list(periods) == sorted(periods)
assert all(p < pd.Period("2025-12", freq="M") for p in periods), (
    "Optuna development folds must end before December 2025"
)
print(f"Tuning {BACKEND} / {TRACK}, mode={CV_MODE}, on {', '.join(DEV_MONTHS)}")

Tuning catboost / off_market, mode=expanding, on 2025-09, 2025-10, 2025-11


## 2 · Starting parameters and search space

The baseline configurations match the explicit benchmark settings in `model_run.ipynb`, including `900` rounds. CatBoost uses Huber on the **list-aware** ratio target to compare with your prior best; its off-market baseline remains RMSE.

We **fix the loss for each study** rather than simultaneously tuning every loss and structural parameter. `n_estimators`/`iterations` is tuned explicitly; no early stopping uses the fold being scored. (Using that fold for both early stopping and scoring would bias selection.)

In [12]:
ESTIMATOR_CLASSES = {
    "catboost": CatBoostRegressor,
    "lightgbm": LightGBMRegressor,
    "xgboost": XGBoostRegressor,
}

BASE_PARAMS = {
    "catboost": {
        "iterations": 900, "depth": 6, "learning_rate": 0.04,
        "l2_leaf_reg": 5.0,
        "loss_function": "Huber:delta=0.05" if TRACK == "list_aware" else "RMSE",
        "random_seed": SEED, "verbose": False, "allow_writing_files": False,
        "task_type": "CPU", "thread_count": -1,
    },
    "lightgbm": {
        "n_estimators": 900, "learning_rate": 0.04,
        "max_depth": 6, "num_leaves": 31, "min_child_samples": 40,
        "reg_lambda": 5.0, "reg_alpha": 0.0,
        "subsample": 0.9, "subsample_freq": 1, "colsample_bytree": 0.9,
        "max_bin": 255, "objective": "regression",
        "random_state": SEED, "n_jobs": -1, "verbosity": -1,
    },
    "xgboost": {
        "n_estimators": 900, "learning_rate": 0.04,
        "max_depth": 6, "min_child_weight": 5.0,
        "reg_lambda": 5.0, "reg_alpha": 0.0,
        "subsample": 0.9, "colsample_bytree": 0.9,
        "tree_method": "hist", "grow_policy": "depthwise",
        "device": "cpu", "enable_categorical": True,
        "objective": "reg:squarederror", "random_state": SEED,
        "n_jobs": -1, "verbosity": 0,
    },
}

# The baseline trial is evaluated first to anchor Optuna against your known setup.
BASELINE_TRIAL = {
    "catboost": {"iterations": 900, "learning_rate": 0.04, "depth": 6,
                 "l2_leaf_reg": 5.0, "random_strength": 1.0},
    "lightgbm": {"n_estimators": 900, "learning_rate": 0.04,
                 "num_leaves": 31, "min_child_samples": 40,
                 "reg_lambda": 5.0, "colsample_bytree": 0.9,
                 "subsample": 0.9},
    "xgboost": {"n_estimators": 900, "learning_rate": 0.04,
                "max_depth": 6, "min_child_weight": 5.0,
                "reg_lambda": 5.0, "colsample_bytree": 0.9,
                "subsample": 0.9},
}


def suggest_params(trial):
    """Only vary parameters relevant to the selected backend; fixed loss and seed."""
    if BACKEND == "lightgbm":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 500, 1800, step=100),
            "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.10, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 15, 63),
            "min_child_samples": trial.suggest_int("min_child_samples", 20, 150, step=10),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 30.0, log=True),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.75, 1.0),
            "subsample": trial.suggest_float("subsample", 0.75, 1.0),
            # Keep depth 6 for a first controlled search; tune it separately later.
            "max_depth": 6,
        }
    if BACKEND == "xgboost":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 500, 1800, step=100),
            "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.10, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 9),
            "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 20.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 30.0, log=True),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.75, 1.0),
            "subsample": trial.suggest_float("subsample", 0.75, 1.0),
        }
    return {
        "iterations": trial.suggest_int("iterations", 500, 1600, step=100),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.10, log=True),
        "depth": trial.suggest_int("depth", 4, 9),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 20.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.1, 3.0, log=True),
    }


def merge_params(overrides):
    params = deepcopy(BASE_PARAMS[BACKEND])
    params.update(overrides)
    return params

print("Fixed loss:", BASE_PARAMS[BACKEND].get("loss_function", BASE_PARAMS[BACKEND].get("objective")))

Fixed loss: RMSE


## 3 · Build the past-only development folds (once)

`fit_transform(train)` is called on sales completed **before** the validation month. `transform(homes)` sees the same inputs as the competition test files via `data.hide_answers`. Each backend's categorical columns are constructed by your existing feature factory.

Caution: in `expanding` mode this caches **three historical training snapshots** in RAM; in `fixed_origin` it reuses one. On your 32 GB machine that is generally reasonable for the current 47 TF-IDF features, but don't reuse this cache for your future 1,000-term experiment without checking memory. Restart the kernel when switching backend/track.

In [14]:
from dataclasses import dataclass

@dataclass
class Fold:
    month: str
    X_train: pd.DataFrame
    y_train: np.ndarray
    X_valid: pd.DataFrame
    valid_log_sold: np.ndarray
    valid_log_list: np.ndarray | None
    target_offset: float
    n_train: int
    ml_num: np.ndarray
    sold_price: np.ndarray


def prepare_folds(listings):
    cached = []
    months_sold = listings["date_sold"].dt.to_period("M")

    # The hidden test uses a frozen model trained through May; this mode mimics
    # the forecasting horizon using a frozen cutoff before the FIRST dev month.
    fixed_train = None
    fixed_provider = None
    fixed_X = None
    if CV_MODE == "fixed_origin":
        fixed_train = listings.loc[listings["date_sold"] < periods[0].start_time]
        if fixed_train.empty:
            raise ValueError("No historical sales before fixed origin")
        fixed_provider = make_features(FEATURE_NAME, model=BACKEND, track=TRACK)
        fixed_X = fixed_provider.fit_transform(fixed_train)
        print(f"Fixed origin: trained feature provider on {len(fixed_train):,} prior sales")

    for period in periods:
        start = time.perf_counter()
        train = (fixed_train if CV_MODE == "fixed_origin" else
                 listings.loc[listings["date_sold"] < period.start_time])
        validation = listings.loc[months_sold == period]
        if len(train) == 0 or len(validation) == 0:
            raise ValueError(f"Empty training or validation data for {period}")

        if CV_MODE == "fixed_origin":
            feature_provider = fixed_provider
            X_train = fixed_X
        else:
            feature_provider = make_features(FEATURE_NAME, model=BACKEND, track=TRACK)
            X_train = feature_provider.fit_transform(train)

        X_valid = feature_provider.transform(data.hide_answers(validation, TRACK))
        if not X_train.columns.equals(X_valid.columns):
            raise ValueError(f"Feature column mismatch for {period}")
        if not X_train.index.equals(train.index) or not X_valid.index.equals(validation.index):
            raise ValueError(f"Feature index mismatch for {period}")

        train_log_sold = np.log(train["sold_price"].to_numpy(dtype=float))
        valid_log_sold = np.log(validation["sold_price"].to_numpy(dtype=float))
        valid_log_list = None
        offset = 0.0
        if TRACK == "list_aware":
            train_log_list = np.log(train["list_price"].to_numpy(dtype=float))
            valid_log_list = np.log(validation["list_price"].to_numpy(dtype=float))
            y_train = train_log_sold - train_log_list
        else:
            if OFF_MARKET_CENTER_TARGET:
                offset = float(np.median(train_log_sold))
            y_train = train_log_sold - offset
        if not np.isfinite(y_train).all() or not np.isfinite(valid_log_sold).all():
            raise ValueError("Nonfinite prices in historical fold")

        cached.append(Fold(
            month=str(period), X_train=X_train, y_train=y_train,
            X_valid=X_valid, valid_log_sold=valid_log_sold,
            valid_log_list=valid_log_list, target_offset=offset,
            n_train=len(train), ml_num=validation["ml_num"].to_numpy(),
            sold_price=validation["sold_price"].to_numpy(dtype=float),
        ))
        feature_mb = (X_train.memory_usage(deep=True).sum() +
                      X_valid.memory_usage(deep=True).sum()) / 1e6
        print(f"{period}: {len(train):,} train / {len(validation):,} validation; "
              f"feature footprint {feature_mb:,.0f} MB; prep {time.perf_counter()-start:,.1f}s")
    return cached

listings = data.load("listings")
CACHED_FOLDS = prepare_folds(listings)
print(f"Cached {len(CACHED_FOLDS)} validation folds; mode={CV_MODE}")

2025-09: 133,779 train / 2,035 validation; feature footprint 80 MB; prep 83.7s
2025-10: 135,814 train / 2,250 validation; feature footprint 81 MB; prep 78.0s
2025-11: 138,064 train / 1,867 validation; feature footprint 83 MB; prep 78.4s
Cached 3 validation folds; mode=expanding


## 4 · Train/score one trial

Uses the repository's `BoostingComponent` wrappers, not native estimator defaults. All cached TF-IDF vocabulary, categorical encodings and comparables remain consistent within each chronological fold.

Objective is **pooled MdAPE**, computed from all development validation homes. Per-month scores and pooled MAPE are recorded for judging the stability of winning trials. A trial is penalized (not silently clipped to valid house prices) for nonfinite or grossly pathological predictions.

In [ ]:
def pct_error_from_log(log_predictions, log_truth):
    log_predictions = np.asarray(log_predictions, dtype=float).reshape(-1)
    log_truth = np.asarray(log_truth, dtype=float).reshape(-1)
    if log_predictions.shape != log_truth.shape:
        raise ValueError("Prediction shape mismatch")
    if (not np.isfinite(log_predictions).all() or
        np.any(log_predictions < -10) or np.any(log_predictions > 40)):
        return None
    # expm1 gives numerically stable percentage errors near zero.
    return 100.0 * np.expm1(log_predictions - log_truth)


def fit_predict_log(fold, params, *, prefit_component=None):
    component = prefit_component
    if component is None:
        component = ESTIMATOR_CLASSES[BACKEND](**params)
        component.fit(fold.X_train, fold.y_train)
    pred = np.asarray(component.predict(fold.X_valid), dtype=float).reshape(-1)
    if TRACK == "off_market":
        return pred + fold.target_offset
    return pred + fold.valid_log_list


def evaluate_params(params, trial=None, *, keep_predictions=False):
    all_errors = []
    per_month = []
    saved_rows = []
    frozen_component = None
    frozen_fit_seconds = 0.0
    if CV_MODE == "fixed_origin":
        fit_start = time.perf_counter()
        frozen_component = ESTIMATOR_CLASSES[BACKEND](**params)
        frozen_component.fit(CACHED_FOLDS[0].X_train, CACHED_FOLDS[0].y_train)
        frozen_fit_seconds = time.perf_counter() - fit_start
    for step, fold in enumerate(CACHED_FOLDS, start=1):
        start = time.perf_counter()
        pred_log = fit_predict_log(fold, params, prefit_component=frozen_component)
        errors = pct_error_from_log(pred_log, fold.valid_log_sold)
        if errors is None:
            if trial is not None:
                trial.set_user_attr("invalid_predictions_month", fold.month)
            return 1e12, [], []
        all_errors.append(errors)
        abs_err = np.abs(errors)
        per_month.append({
            "month": fold.month, "homes": len(errors),
            "mdape": float(np.median(abs_err)),
            "mape": float(np.mean(abs_err)),
            "median_signed": float(np.median(errors)),
            "fit_and_predict_seconds": round(
                time.perf_counter() - start + (frozen_fit_seconds if step == 1 else 0.0), 1
            ),
        })
        if keep_predictions:
            saved_rows.append(pd.DataFrame({
                "ml_num": fold.ml_num, "track": TRACK,
                "month": fold.month, "sold_price": fold.sold_price,
                "predicted_price": np.exp(pred_log),
            }))
        if trial is not None:
            partial_mdape = float(np.median(np.abs(np.concatenate(all_errors))))
            trial.report(partial_mdape, step=step)
            if trial.should_prune():
                raise optuna.TrialPruned(f"Pruned after {fold.month}: MdAPE {partial_mdape:.4f}%")

    pooled = np.concatenate(all_errors)
    score = float(np.median(np.abs(pooled)))
    if trial is not None:
        trial.set_user_attr("pooled_mape", float(np.mean(np.abs(pooled))))
        trial.set_user_attr("per_month", per_month)
    return score, per_month, saved_rows


def objective(trial):
    overrides = suggest_params(trial)
    return evaluate_params(merge_params(overrides), trial=trial)[0]

## 5 · Persistent, resumable Optuna search

Optuna writes completed and pruned trials to `tuning/optuna_studies.db`. Re-running `study.optimize` adds `N_TRIALS` trials. Do **not** run concurrent notebook sessions against the same SQLite study. The study checks its defining configuration on resume so you don't accidentally mix validation windows.

Optuna pruning here happens **only between completed validation folds**, not by peeking at a fold's validation labels to decide the number of trees.

In [ ]:
STUDY_NAME = f"{STUDY_VERSION}_{CV_MODE}_{BACKEND}_{TRACK}_{FEATURE_NAME}"
STUDY_CONFIG = {
    "backend": BACKEND, "track": TRACK, "feature": FEATURE_NAME,
    "dev_months": list(DEV_MONTHS), "validation_mode": CV_MODE,
    "center_off_market": OFF_MARKET_CENTER_TARGET,
    "list_aware_target": LIST_AWARE_TARGET,
    "base_params": BASE_PARAMS[BACKEND],
    "search_version": STUDY_VERSION,
}

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE,
    direction="minimize",
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=8, n_warmup_steps=1),
)
previous_config = study.user_attrs.get("experiment_config")
if previous_config is not None and previous_config != STUDY_CONFIG:
    raise RuntimeError(
        "Study exists with different settings! Increase STUDY_VERSION or use a new study name."
    )
study.set_user_attr("experiment_config", STUDY_CONFIG)
study.enqueue_trial(BASELINE_TRIAL[BACKEND], skip_if_exists=True)

print("Study:", STUDY_NAME)
print("Trials already stored:", len(study.trials))
print("SQLite:", RESULTS_DIR / "optuna_studies.db")
print("Running", N_TRIALS, "additional trials...")
study.optimize(objective, n_trials=N_TRIALS, n_jobs=N_JOBS_STUDIES, gc_after_trial=True)
print("Trials stored after run:", len(study.trials))

[I 2026-10-21 08:27:28,437] A new study created in RDB with name: v1_expanding_catboost_off_market_Xpt_spatiotemporal


Study: v1_expanding_catboost_off_market_Xpt_spatiotemporal
Trials already stored: 1
SQLite: C:\Users\deant\Desktop\Coding\Data Dash Case Comp\tuning\optuna_studies.db
Running 20 additional trials...


[I 2026-10-21 08:30:37,776] Trial 0 finished with value: 9.0624934618446 and parameters: {'iterations': 900, 'learning_rate': 0.04, 'depth': 6, 'l2_leaf_reg': 5.0, 'random_strength': 1.0}. Best is trial 0 with value: 9.0624934618446.
[I 2026-10-21 08:35:18,464] Trial 1 finished with value: 8.327197156727102 and parameters: {'iterations': 900, 'learning_rate': 0.09237421878009823, 'depth': 8, 'l2_leaf_reg': 6.009974718380312, 'random_strength': 0.17000372989211016}. Best is trial 1 with value: 8.327197156727102.
[I 2026-10-21 08:39:13,757] Trial 2 finished with value: 9.545680923169632 and parameters: {'iterations': 600, 'learning_rate': 0.021959816073193316, 'depth': 9, 'l2_leaf_reg': 6.054365855469246, 'random_strength': 1.1114989443094976}. Best is trial 1 with value: 8.327197156727102.
[I 2026-10-21 08:41:50,927] Trial 3 finished with value: 8.673053153806283 and parameters: {'iterations': 500, 'learning_rate': 0.09527257190058894, 'depth': 8, 'l2_leaf_reg': 1.8891200276189386, 'ran

Trials stored after run: 20


## 6 · Examine the best trials and export the winning specification

The best trial is chosen only on the **development** folds. The exported JSON contains the **full** winning parameters, backend, feature, target, and fold-level diagnostics. `MODEL_SPECS` can be passed directly to your current factory.

Do **not** automatically use the December–May validation score to retune the study. Confirm a frozen candidate there sparingly using `model_run.ipynb`.

In [ ]:
from optuna.trial import TrialState

complete = [t for t in study.trials if t.state == TrialState.COMPLETE]
if not complete:
    raise RuntimeError("No completed trials; inspect failed trials or run with N_TRIALS > 0")

leaderboard = pd.DataFrame([{
    "number": t.number,
    "mdape": t.value,
    "mape": t.user_attrs.get("pooled_mape", np.nan),
    "total_fold_seconds": sum(d.get("fit_and_predict_seconds", 0)
                               for d in t.user_attrs.get("per_month", [])),
    **t.params,
} for t in complete]).sort_values(["mdape", "mape"])
display(leaderboard.head(10))

best = study.best_trial
BEST_PARAMS = merge_params(best.params)
BEST_MODEL_SPECS = [(ESTIMATOR_CLASSES[BACKEND], FEATURE_NAME, BEST_PARAMS)]
print("Best Optuna trial:", best.number)
print(f"Development MdAPE: {best.value:.4f}%")
print("Development MAPE:", best.user_attrs.get("pooled_mape"))
print("Per-month scores:")
display(pd.DataFrame(best.user_attrs.get("per_month", [])))
print()
print("Copy this spec into your factory or model_run.ipynb:")
print("MODEL_SPECS = [")
print(f"    ({ESTIMATOR_CLASSES[BACKEND].__name__}, {FEATURE_NAME!r}, {BEST_PARAMS!r}),")
print("]")

artifact = {
    "study_name": STUDY_NAME, "experiment": STUDY_CONFIG,
    "best_trial": best.number,
    "dev_mdape": best.value,
    "dev_mape": best.user_attrs.get("pooled_mape"),
    "monthly_scores": best.user_attrs.get("per_month", []),
    "best_params": BEST_PARAMS,
}
best_path = RESULTS_DIR / f"best_{STUDY_NAME}.json"
best_path.write_text(json.dumps(artifact, indent=2), encoding="utf-8")
print("Saved:", best_path)

,number,mdape,mape,total_fold_seconds,iterations,learning_rate,depth,l2_leaf_reg,random_strength
14,18,8.108889,10.725528,441.6,1400,0.071276,8,4.140531,0.580328
8,10,8.273117,11.006348,277.6,1300,0.086136,6,10.084682,0.450446
9,11,8.317225,10.752156,346.9,1100,0.097314,8,9.059331,0.408021
1,1,8.327197,10.806067,280.6,900,0.092374,8,6.009975,0.170004
12,15,8.402098,10.886574,344.7,1100,0.073083,8,11.614724,1.182162
15,19,8.453639,11.022818,340.3,1600,0.069052,6,5.841176,0.562879
13,16,8.545665,10.992218,275.6,1300,0.085876,6,2.705276,0.644735
11,14,8.550723,11.065330,293.7,1400,0.077278,6,16.147873,0.761966
10,12,8.650836,11.066312,233.4,1100,0.090201,6,6.530675,0.493977
3,3,8.673053,11.146919,157.0,500,0.095273,8,1.889120,0.185600


Best Optuna trial: 18
Development MdAPE: 8.1089%
Development MAPE: 10.725527825397135
Per-month scores:


,month,homes,mdape,mape,median_signed,fit_and_predict_seconds
0,2025-09,2035,8.323866,10.757244,0.645491,144.9
1,2025-10,2250,7.808301,10.481449,0.190897,147.7
2,2025-11,1867,8.324211,10.985107,2.668235,149.0



Copy this spec into your factory or model_run.ipynb:
MODEL_SPECS = [
    (CatBoostRegressor, 'Xpt_spatiotemporal', {'iterations': 1400, 'depth': 8, 'learning_rate': 0.07127609366218435, 'l2_leaf_reg': 4.14053058084107, 'loss_function': 'RMSE', 'random_seed': 42, 'verbose': False, 'allow_writing_files': False, 'task_type': 'CPU', 'thread_count': -1, 'random_strength': 0.580328206016201}),
]
Saved: C:\Users\deant\Desktop\Coding\Data Dash Case Comp\tuning\best_v1_expanding_catboost_off_market_Xpt_spatiotemporal.json


## 7 · Factory integration and optional parity check

This is the **only cell to adapt when you change your production model factory**. For now it matches the present `custom_make_model` behavior: `OffMarketModel(center_target=True)` and `DirectListingModel(target="ratio")`.

The cached training path should give the same predictions as the factory (up to numerical precision) when configured identically. Turn on this check once after a factory refactor; it performs a new **full fit**, so it costs extra runtime.

In [ ]:
def build_factory_model(track, modelspecs):
    """Replace this body with your new model factory call after refactoring."""
    if track == "off_market":
        return OffMarketModel(modelspecs, center_target=OFF_MARKET_CENTER_TARGET)
    if track == "list_aware":
        return DirectListingModel(target=LIST_AWARE_TARGET, modelspecs=modelspecs)
    raise ValueError(track)


RUN_FACTORY_PARITY_CHECK = False
if RUN_FACTORY_PARITY_CHECK:
    fold = CACHED_FOLDS[-1]
    period = pd.Period(fold.month, freq="M")
    train_cutoff = periods[0].start_time if CV_MODE == "fixed_origin" else period.start_time
    train = listings.loc[listings["date_sold"] < train_cutoff]
    valid = listings.loc[listings["date_sold"].dt.to_period("M") == period]
    homes = data.hide_answers(valid, TRACK)

    predicted_by_factory = np.asarray(
        build_factory_model(TRACK, BEST_MODEL_SPECS).fit(train).predict(homes),
        dtype=float,
    )
    predicted_by_cache = np.exp(fit_predict_log(fold, BEST_PARAMS))
    np.testing.assert_allclose(
        predicted_by_factory, predicted_by_cache, rtol=1e-6, atol=0.02
    )
    print("Factory parity PASS:", TRACK, fold.month,
          "max dollar difference", np.max(np.abs(predicted_by_factory - predicted_by_cache)))

## 8 · Optional: export development out-of-time predictions

Useful later for ensemble research. This **refits the best model** on each cached development fold, but doesn't recompute expensive feature engineering. The CSV is development-only; don't optimize a large ensemble on December–May results.

In [ ]:
EXPORT_BEST_DEV_PREDICTIONS = False
if EXPORT_BEST_DEV_PREDICTIONS:
    _, monthly, rows = evaluate_params(BEST_PARAMS, keep_predictions=True)
    preds = pd.concat(rows, ignore_index=True)
    path = RESULTS_DIR / f"oof_{STUDY_NAME}.csv"
    preds.to_csv(path, index=False)
    print("Saved", len(preds), "chronological development predictions to", path)

In [ ]:
LGB_MODEL_SPECS = [
    (LightGBMRegressor, 'Xpt_spatiotemporal', {'n_estimators': 1800, 'learning_rate': 0.052995139011091236, 'max_depth': 6, 'num_leaves': 40, 'min_child_samples': 30, 'reg_lambda': 16.19372143031597, 'reg_alpha': 0.0, 'subsample': 0.8349167453037107, 'subsample_freq': 1, 'colsample_bytree': 0.953443325579757, 'max_bin': 255, 'objective': 'regression', 'random_state': 42, 'n_jobs': -1, 'verbosity': -1}),
]
XGB_MODEL_SPECS = [
    (XGBoostRegressor, 'Xpt_spatiotemporal', {'n_estimators': 700, 'learning_rate': 0.09522271657253033, 'max_depth': 8, 'min_child_weight': 16.684619386293466, 'reg_lambda': 19.50331236061207, 'reg_alpha': 0.0, 'subsample': 0.9804685587557792, 'colsample_bytree': 0.8994749947027713, 'tree_method': 'hist', 'grow_policy': 'depthwise', 'device': 'cpu', 'enable_categorical': True, 'objective': 'reg:squarederror', 'random_state': 42, 'n_jobs': -1, 'verbosity': 0}),
]
CAT_MODEL_SPECS = [
    (CatBoostRegressor, 'Xpt_spatiotemporal', {'iterations': 1400, 'depth': 8, 'learning_rate': 0.07127609366218435, 'l2_leaf_reg': 4.14053058084107, 'loss_function': 'RMSE', 'random_seed': 42, 'verbose': False, 'allow_writing_files': False, 'task_type': 'CPU', 'thread_count': -1, 'random_strength': 0.580328206016201}),
]

## 9 · Confirmation and next runs

1. **Freeze** the selected parameters and `BEST_MODEL_SPECS` after inspecting development months. Optionally run the factory parity check above.
2. Copy `MODEL_SPECS` into `model_run.ipynb` (and update its `custom_make_model` to match this notebook's target settings). Run **December 2025–May 2026 once** for the selected candidate; compare with current results.
3. For another backend or track: change `BACKEND` / `TRACK`, **restart kernel and run all cells**. Each combination has a separate SQLite study.
4. For major feature/factory changes: change `STUDY_VERSION` (and clear kernel) so stale cached matrices or old trials aren't combined with a new experiment.

**Why this notebook deliberately does not use early stopping:** Validation-month labels are used to score each trial, so using those same labels for stopping would introduce additional adaptive selection. Instead, boosting rounds are a search parameter, and Optuna prunes complete chronological folds. Later we can add nested chronological early stopping with a separate earlier stopping set.

**About ensemble tuning:** Export fold-specific out-of-time predictions; fit and select ensemble members on **development** OOF only. Don't treat your existing `r=0.99` correlation threshold as a substitute for independent evaluation.
**Fixed-origin cross-check:** if you choose `CV_MODE="fixed_origin"`, the objective fits **one** model before the first development month and reuses it for subsequent horizons; the optional parity check compares its last-month predictions with your production factory fitted at the same origin.

**Version control:** consider adding `tuning/*.db` and `tuning/*.db-*` to `.gitignore` (SQLite journals and mutable studies). Commit the notebook and selected configuration/results you want to preserve, not every trial database.
